# Подготовка модели

In [3]:
import sys
sys.path.append('../')

import torch
import typing
import json

from src.model import Transformer, generate_batch
from src.dataset import (
    NMRDataset, 
    load_multiplicity_codes, 
    load_split, 
    collate_fn
)

import sentencepiece as spm

from functools import partial

In [4]:
tokenizer = spm.SentencePieceProcessor('../checkpoints/nmr2struct.model')

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
model = Transformer(
    multiplicity_vocab_size=len(NMRDataset.MULTIPLICITY2IDX),
    multiplicity_hidden_dim=48,
    spectrum_d_ff=256,
    spectrum_hidden_dim=104,
    intensity_d_ff=256,
    intensity_hidden_dim=104,
    tgt_vocab_size=tokenizer.vocab_size(),
    d_model=256,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    max_seq_length=400,
    dropout=0.2,
    smiles_pad_token=tokenizer.pad_id()
)
model.load_state_dict(torch.load('../checkpoints/bimodal_256_new_data_full_155.pt', map_location=device))
model = model.to(device)
model.eval()

/tmp/ipykernel_2387047/1523142458.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('../checkpoints/bimodal_256_new_data_full_155.pt', ma

Transformer(
  (spectrum_embedding): SpectrumEmbedding(
    (spectrum_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (intensity_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (multiplicity_embs): ModuleDict(
      (C_NMR): Embedding(10, 48)
      (H_NMR): Embedding(10, 48)
    )
  )
  (decoder_embedding): Embedding(512, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_carbon_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_featu

In [7]:
multiplicity_codes = load_multiplicity_codes('../data/multiplicity_codes.json')

In [16]:
train_data = load_split(
    path='../data/train_full_50k.jsonl', 
    multiplicity_codes=multiplicity_codes,
    solvent='CDCl3'
)

train_dataset = NMRDataset(
    data=train_data,
    solvent='CDCl3',
    tokenizer=tokenizer,
    smiles_bos_id=tokenizer.bos_id(),
    smiles_eos_id=tokenizer.eos_id(),
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    shuffle=False, 
    batch_size=16, 
    collate_fn=partial(
        collate_fn, 
        smiles_pad_id=tokenizer.pad_id(),
        spectrum_pad_token=-1000.,
        max_smiles_len=400
    )
)

Loading train_full_50k.jsonl: 50000it [00:03, 14688.56it/s]
Processing spectra: 100%|██████████| 7940/7940 [00:04<00:00, 1912.45it/s]


In [17]:
# Для каждой молекулы есть разное количество 1H и 13C спектров
# Датасет устроен так что он каждый раз выдает случайную пару спектров
# И с вероятностью 20% маскирует один из спектров
# Если хочется работать иначе с этим - см. src/dataset.py L153

test_data = load_split(
    path='../data/test_full_50k.jsonl', 
    multiplicity_codes=multiplicity_codes,
    solvent='CDCl3' # Так собраны данные
)

test_dataset = NMRDataset(
    data=test_data,
    solvent='CDCl3',
    tokenizer=tokenizer,
    smiles_bos_id=tokenizer.bos_id(),
    smiles_eos_id=tokenizer.eos_id(),
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    shuffle=False, 
    batch_size=16, 
    collate_fn=partial(
        collate_fn, 
        smiles_pad_id=tokenizer.pad_id(),
        spectrum_pad_token=-1000.,
        max_smiles_len=400
    )
)

Loading test_full_50k.jsonl: 50000it [00:02, 17541.31it/s]
Processing spectra: 100%|██████████| 9954/9954 [00:04<00:00, 2301.11it/s]


# На одном батче

In [14]:
data = next(iter(test_loader))

smiles_predictions, saved_hidden_states = generate_batch(
    model=model,
    src_c_nmr=data['C_NMR'],
    src_h_nmr=data['H_NMR'],
    tokenizer=tokenizer,
    max_tokens=400,
    device=device, 
    return_decoder_states=True,
)

In [15]:
decoder_states = saved_hidden_states
# decoder_states: List[Tensor], len = 6, each [B, T, D] = [4, 17, 256]
B = decoder_states[0].size(0)   # 4
T = decoder_states[0].size(1)   # 17
D = decoder_states[0].size(2)   # 256
# stack to [L, B, T, D] = [6, 4, 17, 256]
stacked = torch.stack(decoder_states, dim=0)
# build per-sample list: len = B, each [L, T, D] = [6, 17, 256]
per_sample_states = [stacked[:, b, :, :] for b in range(B)]
# per_sample_states[b].shape -> torch.Size([6, 17, 256])

In [16]:
len(per_sample_states), per_sample_states[0].shape

(4, torch.Size([6, 18, 256]))

In [17]:
true_smiles_list = tokenizer.decode(data['smiles'].squeeze().numpy().tolist())

In [18]:
for pred_smiles, true_smiles in zip(smiles_predictions, tokenizer.decode(data['smiles'].squeeze().numpy().tolist())):
    print(f"True SMILES: {true_smiles}", f"Predicted SMILES: {pred_smiles}", sep='\t')

True SMILES: O=C1OC(=O)c2ccccc21	Predicted SMILES: O=C1c2ccccc2C(=O)N1Cl
True SMILES: Clc1nc(Cl)c2ccccc2n1	Predicted SMILES: O=C(Oc1ccccc1)c1ccc(Cl)nn1
True SMILES: c1ccc2c(N3CCCC3)ncnc2c1	Predicted SMILES: c1ccc2c(N3CCCC3)ncnc2c1
True SMILES: COc1ccc(C=O)cc1OC	Predicted SMILES: COc1ccc(C=O)cc1OC


# На всем датасете

In [18]:
from tqdm.auto import tqdm

def collect_decoder_features_and_smiles(loader, model, tokenizer, device):
    all_features = []
    all_true_smiles = []

    for data in tqdm(loader):
        _, saved_hidden_states = generate_batch(
            model=model,
            src_c_nmr=data['C_NMR'],
            src_h_nmr=data['H_NMR'],
            tokenizer=tokenizer,
            max_tokens=400,
            device=device,
            return_decoder_states=True,
        )

        # saved_hidden_states: list длины L, каждый tensor [B, T, D]
        last_layer = saved_hidden_states[-1]   # [B, T, D]
        pooled = last_layer.mean(dim=1)        # [B, D]

        true_smiles = tokenizer.decode(data['smiles'].cpu().tolist())

        all_features.append(pooled.cpu())
        all_true_smiles.extend(true_smiles)

    X = torch.cat(all_features, dim=0)   # [N, D]
    return X, all_true_smiles

In [19]:
X_train, smiles_train = collect_decoder_features_and_smiles(
    train_loader, model, tokenizer, device
)

X_test, smiles_test = collect_decoder_features_and_smiles(
    test_loader, model, tokenizer, device
)

100%|██████████| 623/623 [13:23<00:00,  1.29s/it]


# Старый блок

In [19]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Fragments

_FRAGMENT_KEYS = sorted(list([k for k, _ in Fragments.fns])) # Сортируем ключи чтобы при каждом запуске порядок был один и тот же (.keys() может возвращать в произвольном порядке)
FRAGMENT_FUNCS = [dict(Fragments.fns)[k] for k in _FRAGMENT_KEYS]

_RING_FEAUTRES_KEYS = [
    'NumRings',
    'NumAliphaticRings',
    'NumAliphaticHeterocycles',
    'NumAliphaticCarbocycles',
    'CalcNumAromaticRings',
    'NumAromaticHeterocycles',
    'NumAromaticCarbocycles',
    'NumHeterocycles' 
]

DESCRIPTOR_KEYS = _FRAGMENT_KEYS + _RING_FEAUTRES_KEYS

def calculate_target(smiles_list: list[str]) -> list[list[float]]:
    result = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            result.append([0.0] * (8 + len(FRAGMENT_FUNCS)))
            continue

        ring_feats = [
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAliphaticCarbocycles(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAromaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAromaticCarbocycles(mol),
            rdMolDescriptors.CalcNumHeterocycles(mol),
        ]

        frag_feats = [func(mol) for func in FRAGMENT_FUNCS]
        res_for_one_smi = ring_feats + frag_feats

        res_for_one_smi = [int(i != 0) for i in res_for_one_smi]
        result.append(res_for_one_smi)
    return result

In [20]:
# assumes `calculate_target(smiles: str) -> list[float]` is defined elsewhere in the notebook

class HiddenStatesProbingDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        hidden_states: typing.List[torch.Tensor] | torch.Tensor,
        smiles_list: typing.List[str],
        hidden_state_idx: int = 0,
        descriptor_idx: int = 0,
    ) -> None:
        """
        hidden_states: list of [d_model] tensors
        smiles_list: list of SMILES strings, same length and order as hidden_states
        """
        hidden_states = [torch.mean(h_i[hidden_state_idx,:,:],0) for h_i in hidden_states] # берём хидден стейт с определённого уровня аттеншена

        
        self.x = torch.stack(hidden_states, dim=0)  # [N, d_model]
        
        self.smiles_list = smiles_list
        self.descriptor_list = [(i[descriptor_idx]) for i in calculate_target(smiles_list)]

    def __len__(self) -> int:
        return self.x.size(0)

    def __getitem__(self, idx: int):
        x = self.x[idx]  # [d_model]

        smiles = self.smiles_list[idx] # for debug simplicity

        desc_values = self.descriptor_list[idx]  
        y = torch.tensor(desc_values, dtype=torch.float32)

        return {"x": x, "y": y, "smiles": smiles}

In [21]:
descriptor_idx = 5
print(DESCRIPTOR_KEYS[descriptor_idx])

test_probbing_dataset = HiddenStatesProbingDataset(
    hidden_states=per_sample_states, # Только 4 примера - для теста. в настоящем эксперименте сюда нужно весь тест передавать
    smiles_list=true_smiles_list,
    descriptor_idx=descriptor_idx
)

test_probbing_loader = torch.utils.data.DataLoader(
    test_probbing_dataset, 
    shuffle=False, 
    batch_size=4
)

fr_Ar_N


In [22]:
class LinearProbe(torch.nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = torch.nn.Linear(input_dim, 2)        # 2 classes: 0 and 1
        self.softmax = torch.nn.Softmax(dim=-1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.fc(x)                      # [B, 2]
        probs = self.softmax(logits)             # [B, 2], sums to 1
        return probs

def train_linear_probe(
    loader: torch.utils.data.DataLoader,
    lr: float = 1e-3,
    epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    
    model = LinearProbe(input_dim=256).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0

        for batch in loader:
            x = batch["x"].to(device)          # [B, d_model]
            y = batch["y"].to(device).long()   # [B]

            optimizer.zero_grad()

            probs = model(x)                   # [B, 2]
            loss = criterion(
                probs.view(-1, 2), # [B, 2]
                y.view(-1) # [B]
            )
            
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            n += x.size(0)

        print(f"Epoch {epoch:02d} | train_loss = {total_loss / n:.4f}")

    return model

In [23]:
probe_model = train_linear_probe(test_probbing_loader, lr=5e-2, epochs=20)

Epoch 01 | train_loss = 0.6997
Epoch 02 | train_loss = 0.5708
Epoch 03 | train_loss = 0.4621
Epoch 04 | train_loss = 0.4011
Epoch 05 | train_loss = 0.3686
Epoch 06 | train_loss = 0.3464
Epoch 07 | train_loss = 0.3331
Epoch 08 | train_loss = 0.3258
Epoch 09 | train_loss = 0.3218
Epoch 10 | train_loss = 0.3194
Epoch 11 | train_loss = 0.3179
Epoch 12 | train_loss = 0.3168
Epoch 13 | train_loss = 0.3161
Epoch 14 | train_loss = 0.3156
Epoch 15 | train_loss = 0.3152
Epoch 16 | train_loss = 0.3149
Epoch 17 | train_loss = 0.3146
Epoch 18 | train_loss = 0.3144
Epoch 19 | train_loss = 0.3143
Epoch 20 | train_loss = 0.3141


# Эксперимент на число углеродов

In [20]:
from rdkit import Chem

def count_carbons(smiles: str) -> int:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)

In [21]:
y_train = torch.tensor([count_carbons(s) for s in smiles_train], dtype=torch.float32)
y_test = torch.tensor([count_carbons(s) for s in smiles_test], dtype=torch.float32)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([7940, 256]) torch.Size([7940])
torch.Size([9954, 256]) torch.Size([9954])


In [22]:
class CarbonCountProbingDataset(torch.utils.data.Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor):
        self.x = X.float()
        self.y = y.float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": self.x[idx],
            "y": self.y[idx],
        }

In [23]:
train_probing_dataset = CarbonCountProbingDataset(X_train, y_train)
test_probing_dataset = CarbonCountProbingDataset(X_test, y_test)

train_probing_loader = torch.utils.data.DataLoader(
    train_probing_dataset,
    shuffle=True,
    batch_size=64
)

test_probing_loader = torch.utils.data.DataLoader(
    test_probing_dataset,
    shuffle=False,
    batch_size=64
)

In [24]:
class LinearProbe(torch.nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = torch.nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x).squeeze(-1)


def train_linear_probe(
    train_loader,
    test_loader,
    lr: float = 1e-3,
    epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    model = LinearProbe(input_dim=256).to(device)
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            x = batch["x"].to(device)
            y = batch["y"].to(device)

            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        preds = []
        targets = []

        with torch.no_grad():
            for batch in test_loader:
                x = batch["x"].to(device)
                y = batch["y"].to(device)

                pred = model(x)

                preds.append(pred.cpu())
                targets.append(y.cpu())

        preds = torch.cat(preds)
        targets = torch.cat(targets)

        mae = torch.mean(torch.abs(preds - targets)).item()

        ss_res = torch.sum((targets - preds) ** 2)
        ss_tot = torch.sum((targets - targets.mean()) ** 2)
        r2 = (1 - ss_res / ss_tot).item()

        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | test_MAE={mae:.4f} | test_R2={r2:.4f}")

    return model

In [25]:
probe_model = train_linear_probe(
    train_probing_loader,
    test_probing_loader,
    lr=1e-3,
    epochs=20,
)

Epoch 01 | train_loss=227.4479 | test_MAE=8.4888 | test_R2=-1.4567
Epoch 02 | train_loss=107.1230 | test_MAE=6.6048 | test_R2=-0.5734
Epoch 03 | train_loss=74.0407 | test_MAE=5.5773 | test_R2=-0.1742
Epoch 04 | train_loss=56.9892 | test_MAE=4.9257 | test_R2=0.0534
Epoch 05 | train_loss=47.3500 | test_MAE=4.5397 | test_R2=0.1881
Epoch 06 | train_loss=41.4142 | test_MAE=4.2788 | test_R2=0.2748
Epoch 07 | train_loss=37.2966 | test_MAE=4.0711 | test_R2=0.3378
Epoch 08 | train_loss=34.1813 | test_MAE=3.8993 | test_R2=0.3917
Epoch 09 | train_loss=31.6956 | test_MAE=3.7518 | test_R2=0.4309
Epoch 10 | train_loss=29.6590 | test_MAE=3.6289 | test_R2=0.4638
Epoch 11 | train_loss=27.9747 | test_MAE=3.5238 | test_R2=0.4895
Epoch 12 | train_loss=26.5577 | test_MAE=3.4298 | test_R2=0.5139
Epoch 13 | train_loss=25.3099 | test_MAE=3.3507 | test_R2=0.5349
Epoch 14 | train_loss=24.2489 | test_MAE=3.2783 | test_R2=0.5518
Epoch 15 | train_loss=23.3230 | test_MAE=3.2184 | test_R2=0.5667
Epoch 16 | train_los